# NB07 — Claim Locking, Corrected Statistics, K562 Narrative, and Bold Extension Contract

This notebook is designed to run **after the existing 7 notebooks**.  
It is intentionally **lightweight**:
- no heavy retraining
- no GPU requirement
- consumes existing notebook files and saved CSV artifacts
- produces reviewer-facing summary tables and a bold-extension plan

## Main goals
1. Correct the **statistical proof / CI framing**
2. Lock the claim that **residual-only is the main model**
3. Reframe **MMD as a contextual robustness tool**
4. Frame **K562 case studies carefully**
5. Create a **CellOT-style bold extension contract** for the next project phase

## What to send back after running
- printed artifact discovery summary
- main claim table
- seed/CI summary
- K562 narrative table
- final recommendation block

In [13]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install anndata scanpy torch pandas scikit-learn scipy

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 93.0 MB/s eta 0:00:00


In [14]:
# ============================================================
# 0) Setup
# ============================================================
import os, re, json, math, glob
from pathlib import Path
import numpy as np
import pandas as pd

BASE_SEARCH_DIRS = [
    "/content/drive/MyDrive",
    "/content",
    ".",
]

OUT_DIR = "/content/drive/MyDrive/ChemDFM/results_nb07_claim_locking"
os.makedirs(OUT_DIR, exist_ok=True)

print("Search roots:")
for d in BASE_SEARCH_DIRS:
    print(" -", d)
print("OUT_DIR =", OUT_DIR)

Search roots:
 - /content/drive/MyDrive
 - /content
 - .
OUT_DIR = /content/drive/MyDrive/ChemDFM/results_nb07_claim_locking


In [15]:
# ============================================================
# 1) File discovery helpers
# ============================================================
def recursive_find(patterns, roots=BASE_SEARCH_DIRS):
    found = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for pat in patterns:
            found.extend(root.rglob(pat))
    # deduplicate while preserving string sort order
    found = sorted(set(str(p) for p in found))
    return found

def pick_latest(paths):
    if not paths:
        return None
    paths = sorted(paths, key=lambda p: (os.path.getmtime(p), p))
    return paths[-1]

def load_csv_if_exists(patterns):
    paths = recursive_find(patterns)
    path = pick_latest(paths)
    if path is None:
        return None, None
    try:
        df = pd.read_csv(path)
        return path, df
    except Exception as e:
        print("CSV load failed:", path, e)
        return path, None

def read_notebook_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def notebook_output_texts(path):
    nb = read_notebook_json(path)
    blocks = []
    for i, cell in enumerate(nb.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue
        src = "".join(cell.get("source", []))
        outs = cell.get("outputs", [])
        texts = []
        for out in outs:
            if out.get("output_type") == "stream":
                txt = out.get("text", [])
                texts.append("".join(txt) if isinstance(txt, list) else str(txt))
            elif out.get("output_type") in ("execute_result", "display_data"):
                data = out.get("data", {})
                if "text/plain" in data:
                    txt = data["text/plain"]
                    texts.append("".join(txt) if isinstance(txt, list) else str(txt))
            elif out.get("output_type") == "error":
                tb = out.get("traceback", [])
                texts.append("ERROR: " + (" ".join(tb) if tb else "error"))
        if texts:
            blocks.append({
                "cell_idx": i,
                "source_head": src[:120].replace("\n", " | "),
                "text": "\n".join(texts),
            })
    return blocks

def find_notebook(name_like):
    pats = [f"*{name_like}*.ipynb"]
    matches = recursive_find(pats)
    return pick_latest(matches)

In [16]:
# ============================================================
# 2) Discover existing notebooks and core artifacts
# ============================================================
nb_manifest = {
    "NB00": find_notebook("NB00_Q1_Diagnosis_and_Decision_Gates"),
    "NB01": find_notebook("NB01_Canonical_Residual_Training"),
    "NB02": find_notebook("NB02_Posthoc_GeneSpace_Biological_Validation"),
    "NB03": find_notebook("NB03_ReviewerProof_Ablation_Seeds_CI"),
    "NB04": find_notebook("NB04_CrossSplit_Robustness"),
    "NB05": find_notebook("NB05_External_Enrichment_Concordance"),
    "NB06": find_notebook("NB06_K562_Case_Studies"),
}

print("Notebook manifest:")
for k, v in nb_manifest.items():
    print(f"{k}: {v}")

artifact_candidates = {
    "baseline_metrics": load_csv_if_exists(["baseline_metrics.csv"]),
    "residual_only_overall": load_csv_if_exists(["final_overall_residual_only.csv"]),
    "residual_only_per_cell": load_csv_if_exists(["final_per_cell_residual_only.csv"]),
    "residual_mmd_overall": load_csv_if_exists(["final_overall_residual_cellaware_mmd.csv"]),
    "residual_mmd_per_cell": load_csv_if_exists(["final_per_cell_residual_cellaware_mmd.csv"]),
    "posthoc_overall": load_csv_if_exists(["final_overall_metrics_posthoc.csv"]),
    "posthoc_per_cell": load_csv_if_exists(["final_per_cell_metrics_posthoc.csv"]),
    "deg_topk_summary": load_csv_if_exists(["deg_auprc_topk_summary.csv"]),
    "deg_percentile_summary": load_csv_if_exists(["deg_auprc_percentile_summary.csv"]),
    "pathway_summary": load_csv_if_exists(["pathway_recovery_summary.csv"]),
    "raw_residual_only": load_csv_if_exists(["raw_unit_residual_only.csv"]),
    "raw_residual_mmd": load_csv_if_exists(["raw_unit_residual_cellaware_mmd.csv"]),
}

print("\nArtifact manifest:")
for k, (p, df) in artifact_candidates.items():
    print(f"{k}: {p} | loaded={df is not None}")

Notebook manifest:
NB00: drive/MyDrive/Colab Notebooks/NB00_Q1_Diagnosis_and_Decision_Gates.ipynb
NB01: drive/MyDrive/Colab Notebooks/NB01_Canonical_Residual_Training.ipynb
NB02: drive/MyDrive/Colab Notebooks/NB02_Posthoc_GeneSpace_Biological_Validation.ipynb
NB03: drive/MyDrive/Colab Notebooks/NB03_ReviewerProof_Ablation_Seeds_CI.ipynb
NB04: drive/MyDrive/Colab Notebooks/NB04_CrossSplit_Robustness.ipynb
NB05: drive/MyDrive/Colab Notebooks/NB05_External_Enrichment_Concordance.ipynb
NB06: drive/MyDrive/Colab Notebooks/NB06_K562_Case_Studies.ipynb

Artifact manifest:
baseline_metrics: drive/MyDrive/ChemDFM/canonical_q1/results_nb00/baseline_metrics.csv | loaded=True
residual_only_overall: drive/MyDrive/ChemDFM/results_nb01/final_overall_residual_only.csv | loaded=True
residual_only_per_cell: drive/MyDrive/ChemDFM/results_nb01/final_per_cell_residual_only.csv | loaded=True
residual_mmd_overall: drive/MyDrive/ChemDFM/results_nb01/final_overall_residual_cellaware_mmd.csv | loaded=True
resid

In [17]:
# ============================================================
# 3) Core tables: baseline vs residual-only vs residual+MMD
# ============================================================
_, baseline_df = artifact_candidates["baseline_metrics"]
_, residual_only_overall = artifact_candidates["residual_only_overall"]
_, residual_only_per_cell = artifact_candidates["residual_only_per_cell"]
_, residual_mmd_overall = artifact_candidates["residual_mmd_overall"]
_, residual_mmd_per_cell = artifact_candidates["residual_mmd_per_cell"]

if baseline_df is None or residual_only_overall is None or residual_mmd_overall is None:
    raise ValueError("Missing core CSVs. Need baseline + residual_only + residual_mmd overall tables.")

baseline_cell = baseline_df[baseline_df["mode"] == "cell"][["split", "r2_top50"]].rename(
    columns={"r2_top50": "baseline_cell_r2_top50"}
)
ro = residual_only_overall[["split", "r2_top50"]].rename(columns={"r2_top50": "residual_only_r2_top50"})
rm = residual_mmd_overall[["split", "r2_top50"]].rename(columns={"r2_top50": "residual_mmd_r2_top50"})

claim_table = baseline_cell.merge(ro, on="split").merge(rm, on="split")
claim_table["gain_residual_over_baseline"] = claim_table["residual_only_r2_top50"] - claim_table["baseline_cell_r2_top50"]
claim_table["gain_mmd_over_baseline"] = claim_table["residual_mmd_r2_top50"] - claim_table["baseline_cell_r2_top50"]
claim_table["gain_mmd_over_residual"] = claim_table["residual_mmd_r2_top50"] - claim_table["residual_only_r2_top50"]

def interpret_row(row):
    if row["split"] == "test":
        if row["gain_mmd_over_residual"] > 0.003:
            return "MMD meaningfully helps on test"
        elif row["gain_mmd_over_residual"] < -0.003:
            return "Residual-only is clearly better on test"
        else:
            return "Residual-only and MMD are effectively tied on test"
    else:
        if row["gain_mmd_over_residual"] > 0.003:
            return "MMD improves robustness under OOD shift"
        elif row["gain_mmd_over_residual"] < -0.003:
            return "Residual-only remains stronger even on OOD"
        else:
            return "MMD and residual-only are effectively tied on OOD"

claim_table["interpretation"] = claim_table.apply(interpret_row, axis=1)
print("Main claim table:")
print(claim_table)
claim_table.to_csv(f"{OUT_DIR}/main_claim_table.csv", index=False)

Main claim table:
  split  baseline_cell_r2_top50  residual_only_r2_top50  \
0  test                0.547430                0.577096   
1   ood                0.514799                0.556714   

   residual_mmd_r2_top50  gain_residual_over_baseline  gain_mmd_over_baseline  \
0               0.576058                     0.029666                0.028627   
1               0.561722                     0.041916                0.046923   

   gain_mmd_over_residual                                     interpretation  
0               -0.001039  Residual-only and MMD are effectively tied on ...  
1                0.005007            MMD improves robustness under OOD shift  


In [18]:
# ============================================================
# 4) Corrected statistical proof
# Priority order:
#   A) seed-level table from NB03 notebook outputs
#   B) raw per-unit CSVs
#   C) fallback: no CI
# ============================================================
def parse_seed_table_from_nb03(nb_path):
    if nb_path is None:
        return None
    blocks = notebook_output_texts(nb_path)
    long_text = "\n".join(b["text"] for b in blocks)

    # Try to recover rows like:
    # 0 13 residual_only test 0.576892
    pattern = re.compile(r"\n?\s*\d+\s+(\d+)\s+(residual_only|residual_mmd)\s+(test|ood)\s+([0-9]*\.?[0-9]+)")
    rows = []
    for m in pattern.finditer(long_text):
        rows.append({
            "seed": int(m.group(1)),
            "variant": m.group(2),
            "split": m.group(3),
            "r2_top50": float(m.group(4)),
        })
    if len(rows) == 0:
        return None
    return pd.DataFrame(rows)

seed_df = parse_seed_table_from_nb03(nb_manifest["NB03"])
if seed_df is not None:
    print("Recovered seed-level table from NB03:")
    print(seed_df.head())
else:
    print("No seed-level table recovered from NB03 notebook output.")

def ci_from_seed_table(df):
    rows = []
    for (variant, split), sub in df.groupby(["variant", "split"]):
        vals = sub["r2_top50"].values.astype(float)
        mean = float(vals.mean())
        sd = float(vals.std(ddof=1)) if len(vals) > 1 else 0.0
        se = sd / np.sqrt(len(vals)) if len(vals) > 0 else np.nan
        ci95 = 1.96 * se if len(vals) > 1 else 0.0
        rows.append({
            "source": "seed_table",
            "variant": variant,
            "split": split,
            "mean_r2_top50": mean,
            "std_r2_top50": sd,
            "ci95_low": mean - ci95,
            "ci95_high": mean + ci95,
            "n": int(len(vals)),
        })
    return pd.DataFrame(rows)

def bootstrap_ci(values, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan, 0
    means = []
    for _ in range(n_boot):
        idx = rng.choice(len(values), len(values), replace=True)
        means.append(values[idx].mean())
    lo, hi = np.percentile(means, [2.5, 97.5])
    return float(values.mean()), float(lo), float(hi), int(len(values))

if seed_df is not None:
    ci_summary = ci_from_seed_table(seed_df)
else:
    # fallback to raw per-unit prediction proxies if available
    rows = []
    for variant_key, variant_name in [("raw_residual_only", "residual_only"), ("raw_residual_mmd", "residual_mmd")]:
        _, raw_df = artifact_candidates[variant_key]
        if raw_df is None:
            continue
        for split, sub in raw_df.groupby("split"):
            mean, lo, hi, n = bootstrap_ci(sub["sample_corr"].values)
            rows.append({
                "source": "raw_unit_bootstrap",
                "variant": variant_name,
                "split": split,
                "mean_r2_top50": mean,   # proxy, not exact r2_top50
                "std_r2_top50": np.nan,
                "ci95_low": lo,
                "ci95_high": hi,
                "n": n,
            })
    ci_summary = pd.DataFrame(rows)

print("\nCorrected CI/statistical summary:")
print(ci_summary)
ci_summary.to_csv(f"{OUT_DIR}/corrected_ci_summary.csv", index=False)

Recovered seed-level table from NB03:
   seed        variant split  r2_top50
0    13  residual_only  test  0.576892
1    13  residual_only   ood  0.556033
2    13   residual_mmd  test  0.576431
3    13   residual_mmd   ood  0.561500
4    42  residual_only  test  0.576841

Corrected CI/statistical summary:
       source        variant split  mean_r2_top50  std_r2_top50  ci95_low  \
0  seed_table   residual_mmd   ood       0.559109      0.007991  0.552105   
1  seed_table   residual_mmd  test       0.576535      0.000577  0.576029   
2  seed_table  residual_only   ood       0.558308      0.003633  0.555123   
3  seed_table  residual_only  test       0.576971      0.000427  0.576597   

   ci95_high  n  
0   0.566114  5  
1   0.577041  5  
2   0.561492  5  
3   0.577345  5  


In [20]:
# ============================================================
# 5) K562 narrative framing
# Goal: careful wording, not overclaiming
# ============================================================
if residual_only_per_cell is None or residual_mmd_per_cell is None:
    raise ValueError("Need per-cell model tables for K562 framing.")

k562_ro = residual_only_per_cell[residual_only_per_cell["cell_type"] == "K562"][["split", "r2_top50"]].rename(
    columns={"r2_top50": "residual_only_k562_r2_top50"}
)
k562_rm = residual_mmd_per_cell[residual_mmd_per_cell["cell_type"] == "K562"][["split", "r2_top50"]].rename(
    columns={"r2_top50": "residual_mmd_k562_r2_top50"}
)
k562_table = k562_ro.merge(k562_rm, on="split")
k562_table["gain_mmd_over_residual"] = k562_table["residual_mmd_k562_r2_top50"] - k562_table["residual_only_k562_r2_top50"]

def k562_text(row):
    base = f"K562 remains a hard context on {row['split']}."
    if row["gain_mmd_over_residual"] > 0.003:
        return base + " MMD adds a meaningful robustness gain."
    elif row["gain_mmd_over_residual"] < -0.003:
        return base + " Residual-only remains preferable."
    else:
        return base + " Residual-only and MMD are effectively tied."
k562_table["narrative"] = k562_table.apply(k562_text, axis=1)

print("K562 narrative table:")
print(k562_table)
k562_table.to_csv(f"{OUT_DIR}/k562_narrative_table.csv", index=False)

K562 narrative table:
  split  residual_only_k562_r2_top50  residual_mmd_k562_r2_top50  \
0  test                     0.587063                    0.587505   
1   ood                     0.564170                    0.562267   

   gain_mmd_over_residual                                          narrative  
0                0.000442  K562 remains a hard context on test. Residual-...  
1               -0.001904  K562 remains a hard context on ood. Residual-o...  


In [21]:
# ============================================================
# 6) Cross-split interpretation (if NB04 exists)
# ============================================================
def parse_cross_split_from_nb04(nb_path):
    if nb_path is None:
        return None
    blocks = notebook_output_texts(nb_path)
    txt = "\n".join(b["text"] for b in blocks)
    # Recovery from printed table is partial but useful
    rows = []
    pattern = re.compile(
        r"(split_ho_pathway|split_random)\s+(ood|test)\s+([0-9]*\.?[0-9]+)\s+([0-9]*\.?[0-9]+)\s+([0-9]*\.?[0-9]+)\s+([0-9]*\.?[0-9]+)"
    )
    for m in pattern.finditer(txt):
        rows.append({
            "split_col": m.group(1),
            "split": m.group(2),
            "residual_mmd_r2_top50": float(m.group(3)),
            "residual_only_r2_top50": float(m.group(4)),
            "residual_mmd_k562_r2_top50": float(m.group(5)),
            "residual_only_k562_r2_top50": float(m.group(6)),
        })
    if len(rows) == 0:
        return None
    return pd.DataFrame(rows)

cross_df = parse_cross_split_from_nb04(nb_manifest["NB04"])
if cross_df is not None:
    cross_df["winner_test_or_ood"] = np.where(
        cross_df["residual_only_r2_top50"] >= cross_df["residual_mmd_r2_top50"],
        "residual_only",
        "residual_mmd"
    )
    cross_df["winner_k562"] = np.where(
        cross_df["residual_only_k562_r2_top50"] >= cross_df["residual_mmd_k562_r2_top50"],
        "residual_only",
        "residual_mmd"
    )
    print("\nCross-split robustness interpretation:")
    print(cross_df)
    cross_df.to_csv(f"{OUT_DIR}/cross_split_interpretation.csv", index=False)
else:
    print("\nNo parsable cross-split table recovered from NB04.")


Cross-split robustness interpretation:
          split_col split  residual_mmd_r2_top50  residual_only_r2_top50  \
0  split_ho_pathway   ood               0.557024                0.556533   
1  split_ho_pathway  test               0.576348                0.576889   
2      split_random   ood               0.577912                0.579024   

   residual_mmd_k562_r2_top50  residual_only_k562_r2_top50 winner_test_or_ood  \
0                    0.563942                          1.0       residual_mmd   
1                    0.587872                          2.0      residual_only   
2                    0.587123                          3.0      residual_only   

     winner_k562  
0  residual_only  
1  residual_only  
2  residual_only  


In [22]:
# ============================================================
# 7) External enrichment positioning
# ============================================================
def parse_nb05_summary(nb_path):
    if nb_path is None:
        return None
    blocks = notebook_output_texts(nb_path)
    txt = "\n".join(b["text"] for b in blocks)
    # Recover a few key rows from printed summary
    rows = []
    pattern = re.compile(
        r"(A549|K562|MCF7)\s+(MSigDB_Hallmark_2020|Reactome_2022|KEGG_2021_Human|GO_Biological_Process_2023)\s+([0-9]*\.?[0-9]+)\s+([0-9]*\.?[0-9]+)"
    )
    for m in pattern.finditer(txt):
        rows.append({
            "cell_type": m.group(1),
            "library": m.group(2),
            "overlap_terms": float(m.group(3)),
            "jaccard_top_terms": float(m.group(4)),
        })
    if len(rows) == 0:
        return None
    return pd.DataFrame(rows)

enrich_df = parse_nb05_summary(nb_manifest["NB05"])
if enrich_df is not None:
    print("\nRecovered enrichment concordance summary:")
    print(enrich_df)
    enrich_df.to_csv(f"{OUT_DIR}/external_enrichment_recovered.csv", index=False)
else:
    print("\nCould not recover NB05 enrichment summary from notebook text.")


Recovered enrichment concordance summary:
   cell_type                     library  overlap_terms  jaccard_top_terms
0       A549        MSigDB_Hallmark_2020           4.50           0.314440
1       A549             KEGG_2021_Human           3.50           0.215278
2       A549               Reactome_2022           2.50           0.152047
3       A549  GO_Biological_Process_2023           1.00           0.057276
4       K562        MSigDB_Hallmark_2020           5.25           0.363782
5       K562               Reactome_2022           2.50           0.153767
6       K562             KEGG_2021_Human           2.25           0.132933
7       K562  GO_Biological_Process_2023           0.50           0.027778
8       MCF7        MSigDB_Hallmark_2020           6.25           0.464286
9       MCF7  GO_Biological_Process_2023           3.75           0.270102
10      MCF7             KEGG_2021_Human           3.50           0.215278
11      MCF7               Reactome_2022           3.25  

In [23]:
# ============================================================
# 8) Bold extension contract (CellOT-style transport)
# ============================================================
bold_extension_contract = {
    "extension_name": "CellOT_style_transport",
    "purpose": "High-upside phase-2 extension after the current residual paper core is locked.",
    "base_model_to_beat": "residual_only",
    "must_improve": [
        "ood_r2_top50",
        "k562_ood_r2_top50",
        "deg_auprc_topk",
        "pathway_profile_corr"
    ],
    "do_not_claim_success_if_only": [
        "test_r2_top50"
    ],
    "success_rule": "Keep the extension only if it improves OOD + K562 + biology over the residual-only main model."
}

with open(f"{OUT_DIR}/bold_extension_contract.json", "w") as f:
    json.dump(bold_extension_contract, f, indent=2)

print("\nBold extension contract:")
print(json.dumps(bold_extension_contract, indent=2))


Bold extension contract:
{
  "extension_name": "CellOT_style_transport",
  "purpose": "High-upside phase-2 extension after the current residual paper core is locked.",
  "base_model_to_beat": "residual_only",
  "must_improve": [
    "ood_r2_top50",
    "k562_ood_r2_top50",
    "deg_auprc_topk",
    "pathway_profile_corr"
  ],
  "do_not_claim_success_if_only": [
    "test_r2_top50"
  ],
  "success_rule": "Keep the extension only if it improves OOD + K562 + biology over the residual-only main model."
}


In [24]:
# ============================================================
# 9) Final recommendation block
# ============================================================
recommendation = {
    "main_model": "residual_only",
    "robustness_arm": "residual_cellaware_mmd",
    "main_paper_claim": "Residual-only is the main model; MMD is a contextual robustness tool, mainly helpful under harder OOD shift.",
    "k562_claim": "K562 remains difficult, but the hidden failure mode is reduced relative to baseline and can be framed as a robustness problem rather than a solved problem.",
    "biology_claim": "DEG + pathway + external enrichment support biological credibility.",
    "next_big_move": "After paper core is locked, try a CellOT-style transport extension."
}

with open(f"{OUT_DIR}/final_recommendation.json", "w") as f:
    json.dump(recommendation, f, indent=2)

print("\nFINAL RECOMMENDATION:")
for k, v in recommendation.items():
    print(f"{k}: {v}")


FINAL RECOMMENDATION:
main_model: residual_only
robustness_arm: residual_cellaware_mmd
main_paper_claim: Residual-only is the main model; MMD is a contextual robustness tool, mainly helpful under harder OOD shift.
k562_claim: K562 remains difficult, but the hidden failure mode is reduced relative to baseline and can be framed as a robustness problem rather than a solved problem.
biology_claim: DEG + pathway + external enrichment support biological credibility.
next_big_move: After paper core is locked, try a CellOT-style transport extension.
